# Part 3 - Price-History Analytics (4-credit only)

> Case study part: Part 3 of 3
> Required for: 4-credit (BDI 593) students only
> Role: You are a Real Estate Analyst at Quad Capital Partners (QCP).

In Parts 1 and 2 we treated each listing as a single, static row. But Zillow
also publishes a price-history event log - every list, price change,
contingent, pending, sold, and removal event for each property. That event
log is what an experienced real-estate analyst uses to answer the questions
that actually drive acquisitions:

- How long do homes typically take to sell in each market?
- How often do sellers cut their list price, and by how much?
- What is the typical list-to-sale discount?
- Can we predict the realized sale price from the list price plus what we
  see in the early days of a listing?

This notebook reshapes the event log into property-level features and uses
them - combined with the structured features from Part 2a and the NLP
features from Part 2b - to predict a realized sale outcome, not just a
list price.


## ⚙️ Setup


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBRegressor

pd.set_option("display.max_columns", 80)


## 📥 Load All Three Inputs

- `properties_clean.parquet` from Part 1
- `nlp_features.parquet` from Part 2b
- `price_history.parquet` (raw, this folder)


In [ ]:
DATA_DIR = "data"

properties = pd.read_parquet(f"{DATA_DIR}/properties_clean.parquet")
nlp_pack   = pd.read_parquet(f"{DATA_DIR}/nlp_features.parquet")
history    = pd.read_parquet(f"{DATA_DIR}/price_history.parquet")

print("properties:", properties.shape)
print("nlp_pack:  ", nlp_pack.shape)
print("history:   ", history.shape)
history.head()


## 🩺 What Is in the Event Log?


In [ ]:
event_counts = history["event_type"].value_counts().sort_values().reset_index()
event_counts.columns = ["event_type", "count"]
fig = px.bar(
    event_counts,
    x="count", y="event_type", orientation="h",
    title="Volume of each event type in the Zillow price history",
    labels={"count": "Events recorded", "event_type": ""},
    height=400,
)
fig.show()


:::{seealso} Event-type cheat sheet

- `listedForSale` - a fresh listing.
- `priceChange` - seller cut or raised the asking price.
- `contingent` - offer accepted, contingent on financing/inspection/etc.
- `pendingSale` - under contract, sale expected.
- `sold` - closed sale.
- `listingRemoved` - pulled from the market without selling.
- `listedForRent` - switched from for-sale to for-rent (signal for
  unsuccessful sale).
:::


## 🔁 Reconstruct Listing Cycles per Property

Many properties have multiple list -> sold cycles. We define a "cycle" as
the events between successive `listedForSale` rows for the same property,
ending in either a `sold` or a `listingRemoved`.


Number each cycle by counting fresh listings per property.

In [ ]:
hist = history.sort_values(["property_id", "date"]).copy()

hist["is_listing"] = (hist["event_type"] == "listedForSale").astype(int)
hist["cycle_id"]   = hist.groupby("property_id")["is_listing"].cumsum()

# Drop events that occur before the first listing on record (rare).
hist = hist[hist["cycle_id"] >= 1]


Distribution of cycles per property.

In [ ]:
cycles_n = hist.groupby("property_id")["cycle_id"].max()
fig = px.histogram(
    cycles_n.clip(upper=5),
    nbins=5,
    title="Number of list->sold cycles per property (capped at 5)",
    labels={"value": "Cycles per property"},
)
fig.update_layout(showlegend=False)
fig.show()


## 🧮 Aggregate Each Cycle Into One Row

For every cycle, we want:

- the original list price and date,
- whether and when it sold,
- the realized sale price,
- the list-to-sale discount,
- how many price cuts happened during the cycle,
- and the cumulative depth of those price cuts.


Define the per-cycle reducer.

In [ ]:
def cycle_summary(g):
    g = g.sort_values("date")
    list_rows = g[g["event_type"] == "listedForSale"]
    sold_rows = g[g["event_type"] == "sold"]
    pending_rows = g[g["event_type"] == "pendingSale"]
    cuts = g[(g["event_type"] == "priceChange") & (g["price_change_rate"] < 0)]

    list_date  = list_rows["date"].min() if len(list_rows) else pd.NaT
    list_price = list_rows["price"].iloc[0] if len(list_rows) else np.nan
    sold_date  = sold_rows["date"].max() if len(sold_rows) else pd.NaT
    sold_price = sold_rows["price"].iloc[-1] if len(sold_rows) else np.nan
    pending_date = pending_rows["date"].min() if len(pending_rows) else pd.NaT

    days_to_pending = (pending_date - list_date).days if pd.notnull(list_date) and pd.notnull(pending_date) else np.nan
    days_to_sold    = (sold_date    - list_date).days if pd.notnull(list_date) and pd.notnull(sold_date) else np.nan

    return pd.Series({
        "list_date": list_date,
        "list_price": list_price,
        "sold_date": sold_date,
        "sold_price": sold_price,
        "days_to_pending": days_to_pending,
        "days_to_sold": days_to_sold,
        "n_price_cuts": len(cuts),
        "total_pct_cut": cuts["price_change_rate"].sum() * 100,  # negative number
        "ended_sold": int(len(sold_rows) > 0),
        "ended_removed": int((g["event_type"] == "listingRemoved").any() and len(sold_rows) == 0),
    })


Apply the reducer and compute the realized list-to-sale discount.

In [ ]:
cycle_df = (
    hist.groupby(["property_id", "cycle_id"], as_index=False)
        .apply(cycle_summary)
        .reset_index(drop=True)
)
cycle_df["list_to_sale_pct"] = (
    (cycle_df["sold_price"] - cycle_df["list_price"]) / cycle_df["list_price"] * 100
)
print(f"{len(cycle_df):,} listing cycles reconstructed")
cycle_df.head()


## 📈 Market Dynamics - Visualizations for the Deck


Filter to plausible sold cycles and join in market and campus context.

In [ ]:
sold_cycles = cycle_df.dropna(subset=["sold_price", "list_price", "days_to_sold"]).copy()
sold_cycles = sold_cycles[(sold_cycles["days_to_sold"].between(0, 365 * 2))
                          & (sold_cycles["list_price"] > 10_000)
                          & (sold_cycles["sold_price"] > 10_000)]

market_lookup = (
    properties[["property_id", "market", "campus", "address_state"]]
    .drop_duplicates("property_id")
)
sold_cycles = sold_cycles.merge(market_lookup, on="property_id", how="inner")
print(f"{len(sold_cycles):,} sold cycles with usable market dynamics")


In [ ]:
fig = px.box(
    sold_cycles, x="market", y="days_to_sold",
    category_orders={"market": sorted(sold_cycles["market"].unique())},
    title="Days from list to sale, by market",
    labels={"market": "", "days_to_sold": "Days from list to sale"},
    height=500,
)
fig.update_yaxes(range=[0, 300])
fig.update_xaxes(tickangle=-30)
fig.show()


In [ ]:
fig = px.box(
    sold_cycles, x="market", y="list_to_sale_pct",
    category_orders={"market": sorted(sold_cycles["market"].unique())},
    title="List-to-sale price change, by market (%)",
    labels={"market": "", "list_to_sale_pct": "(sale - list) / list × 100%"},
    height=500,
)
fig.add_hline(y=0, line_dash="dot")
fig.update_yaxes(range=[-30, 30])
fig.update_xaxes(tickangle=-30)
fig.show()


:::{tip} Investor reading

Markets where the box plot sits above zero are seller's markets - homes
sell for more than ask. Markets that sit below zero are buyer's markets -
most homes sell at a discount. This single chart often replaces a
twenty-page market report.
:::


In [ ]:
fig = px.histogram(
    sold_cycles[sold_cycles["n_price_cuts"] <= 6],
    x="n_price_cuts", color="ended_sold",
    title="Price cuts before sale (cycles ending sold vs. removed)",
    labels={"n_price_cuts": "Number of price cuts in the cycle"},
    barmode="group",
)
fig.show()


Quarterly sale volume to confirm we have enough recent data to model.

In [ ]:
sold_cycles["sold_quarter"] = sold_cycles["sold_date"].dt.to_period("Q").dt.to_timestamp()
ts = sold_cycles.groupby("sold_quarter", as_index=False).size()
fig = px.line(
    ts, x="sold_quarter", y="size",
    title="Closed sales per quarter (campustown footprint)",
    labels={"sold_quarter": "", "size": "Closed sales"},
    markers=True,
)
fig.show()


## 🎯 Re-frame the Modeling Problem

In Part 2 we predicted list price. Now we have something more valuable:
the realized sale price. The Investment Committee actually cares about
sale price, not list price, because that is what QCP ends up paying.

We will:

1. Take only the most-recent sold cycle per property (so each property
   appears at most once).
2. Join in the structured features from Part 1 and the NLP features from
   Part 2b.
3. Engineer history-derived features that would have been observable
   at the moment of listing (so we don't leak future information).
4. Train a model that predicts `sold_price` and compare it to a
   list-price-only baseline.


Keep only the most recent sold cycle per property.

In [ ]:
latest_sold = (
    sold_cycles.sort_values(["property_id", "sold_date"])
               .groupby("property_id", as_index=False)
               .tail(1)
)
print(f"{len(latest_sold):,} properties with a most-recent sold cycle")


:::{caution} Avoiding target leakage

`days_to_sold`, `n_price_cuts`, and `total_pct_cut` are only known after
the sale closes. If we feed them into a model that predicts the sale price
of a future listing, we are cheating - those features are unobservable at
list time.

Instead, we engineer historical priors per market and per property:

- the historical median list-to-sale discount in this property's market,
- the historical median days-to-sale in this property's market,
- the property's own prior list price (if it has previously been listed).

These are all observable at list time and are honest features.
:::


Split the cycles into a historical "prior" pool and a recent "target" pool,
then compute per-market priors from the historical pool only.


In [ ]:
HORIZON = pd.Timestamp("2024-01-01")

prior_pool = sold_cycles[sold_cycles["sold_date"] < HORIZON]
target_pool = latest_sold[latest_sold["sold_date"] >= HORIZON].copy()

market_priors = (
    prior_pool.groupby("market")
              .agg(market_hist_disc=("list_to_sale_pct", "median"),
                   market_hist_days=("days_to_sold", "median"),
                   market_hist_cuts=("n_price_cuts", "mean"))
              .reset_index()
)
target_pool = target_pool.merge(market_priors, on="market", how="left")
print(f"Prior pool: {len(prior_pool):,} historical cycles")
print(f"Target pool (>= {HORIZON.date()}): {len(target_pool):,} cycles to model")
target_pool.head(3)


## 🧱 Assemble the Final Modeling Frame


Pull the structural features from Part 1. We deliberately drop
`market`, `campus`, and `address_state` here because `target_pool` already
carries them via the earlier `market_lookup` merge - if we let the same
columns appear on both sides of this merge they would get suffixed to
`address_state_x` / `address_state_y` and break later feature lookups.


In [ ]:
struct_cols = [
    "property_id", "beds", "baths", "living_area", "lot_size",
    "year_built", "reso_parking_capacity", "latitude", "longitude",
    "address_city", "address_zipcode",
    "reso_architectural_style",
]
struct = properties[struct_cols].drop_duplicates("property_id").copy()
struct["address_zipcode"] = struct["address_zipcode"].astype(str)


In [ ]:
frame = (
    target_pool.merge(struct, on="property_id", how="inner")
               .merge(nlp_pack, on="property_id", how="left")
)
print(f"{len(frame):,} rows in the final modeling frame")
print("columns:", list(frame.columns)[:14], "...")


Define the numeric and categorical feature lists for the model.

In [ ]:
NUMERIC_FEATURES = [
    # Structural
    "beds", "baths", "living_area", "lot_size", "year_built",
    "reso_parking_capacity", "latitude", "longitude",
    # Listing-time signal
    "list_price",
    # Market priors (history-derived but leakage-safe)
    "market_hist_disc", "market_hist_days", "market_hist_cuts",
    # NLP (numeric)
    "n_chars", "n_words",
] + [c for c in frame.columns if c.startswith("amen_")]

CATEGORICAL_FEATURES = [
    "address_state", "address_city", "address_zipcode",
    "campus",
    "reso_architectural_style",
]
TARGET = "sold_price"


Build the X / y matrices and the train/test split.

In [ ]:
X = frame[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
y = frame[TARGET].copy()
y_log = np.log1p(y)

X_train, X_test, y_train_log, y_test_log, y_train_raw, y_test_raw = train_test_split(
    X, y_log, y,
    test_size=0.20, random_state=42,
    stratify=frame["address_state"],
)
print(f"Train: {len(X_train):,}   Test: {len(X_test):,}")


## 🔧 Two Models, One Comparison Table

- List-price-only baseline: predict `sold_price` from `list_price` and
  the structural features only. This is the model a lazy AVM would use.
- History-aware model: add the leakage-safe market priors and the NLP
  features.


Pipeline factory shared by both models.

In [ ]:
def make_pipeline(num_cols, cat_cols):
    num = Pipeline([("impute", SimpleImputer(strategy="median"))])
    cat = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
    ])
    pre = ColumnTransformer([("num", num, num_cols), ("cat", cat, cat_cols)])
    return Pipeline([
        ("preprocess", pre),
        ("regressor", XGBRegressor(
            n_estimators=600, max_depth=6, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9,
            random_state=42, n_jobs=-1, tree_method="hist",
        )),
    ])


Fit the list-price-only baseline.

In [ ]:
BASELINE_NUM = [
    "beds", "baths", "living_area", "lot_size", "year_built",
    "reso_parking_capacity", "latitude", "longitude", "list_price",
]
baseline_pipe = make_pipeline(BASELINE_NUM, CATEGORICAL_FEATURES)
baseline_pipe.fit(X_train, y_train_log)


Fit the history-aware model on the full feature set.

In [ ]:
history_pipe = make_pipeline(NUMERIC_FEATURES, CATEGORICAL_FEATURES)
history_pipe.fit(X_train, y_train_log)


Compare the two on the held-out test set.

In [ ]:
def metrics(y_true, y_pred):
    return {
        "MAE":  mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2":   r2_score(y_true, y_pred),
        "MAPE": np.mean(np.abs((y_true - y_pred) / y_true)) * 100,
    }

baseline_pred = np.expm1(baseline_pipe.predict(X_test))
history_pred  = np.expm1(history_pipe.predict(X_test))

comparison = pd.DataFrame({
    "List-price-only (structural)": metrics(y_test_raw, baseline_pred),
    "History-aware (priors + NLP)": metrics(y_test_raw, history_pred),
}).T
comparison


## 🔬 Where Does the History-Aware Model Add Value?


In [ ]:
diag = pd.DataFrame({
    "actual": y_test_raw.values,
    "list_price": X_test["list_price"].values,
    "baseline": baseline_pred,
    "history": history_pred,
    "market": frame.loc[X_test.index, "market"].values,
    "campus": frame.loc[X_test.index, "campus"].values,
})
diag["baseline_pct_err"] = (diag["actual"] - diag["baseline"]).abs() / diag["actual"] * 100
diag["history_pct_err"]  = (diag["actual"] - diag["history"]).abs()  / diag["actual"] * 100
diag["improvement_pp"]   = diag["baseline_pct_err"] - diag["history_pct_err"]
diag["actual_disc_pct"]  = (diag["actual"] - diag["list_price"]) / diag["list_price"] * 100

by_market = (
    diag.groupby("market", as_index=False)
        .agg(n=("actual", "count"),
             baseline=("baseline_pct_err", "mean"),
             history=("history_pct_err", "mean"),
             improvement_pp=("improvement_pp", "mean"))
        .sort_values("improvement_pp", ascending=False)
)
by_market.round(2)


In [ ]:
fig = px.bar(
    by_market.sort_values("improvement_pp"),
    x="improvement_pp", y="market", orientation="h",
    color="improvement_pp", color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    title="History-aware model - reduction in mean |% error| vs. list-price baseline",
    labels={"improvement_pp": "Δ MAPE (baseline - history), pp", "market": ""},
    height=600,
)
fig.show()


## 💼 A Bonus Model - Predicting the Discount Itself

For QCP, the most actionable number is not the predicted sale price but the
expected discount off list price. If our model predicts that homes in
Athens, GA, sell on average 4% below ask but a particular listing is priced
3% above the model's predicted sale value, the offer should reflect that.


Compute the discount target and fit a discount-prediction model.

In [ ]:
y_disc_train = (np.expm1(y_train_log) - X_train["list_price"]) / X_train["list_price"]
y_disc_test  = (y_test_raw - X_test["list_price"]) / X_test["list_price"]

disc_pipe = make_pipeline(
    [c for c in NUMERIC_FEATURES if c != "list_price"],   # exclude raw list price level
    CATEGORICAL_FEATURES,
)
disc_pipe.fit(X_train, y_disc_train)
disc_pred = disc_pipe.predict(X_test)


In [ ]:
print(f"Discount-prediction MAE:  {mean_absolute_error(y_disc_test, disc_pred) * 100:.2f} pp")
print(f"Discount-prediction RMSE: {np.sqrt(mean_squared_error(y_disc_test, disc_pred)) * 100:.2f} pp")
print(f"Discount-prediction R²:   {r2_score(y_disc_test, disc_pred):.3f}")


In [ ]:
fig = px.scatter(
    pd.DataFrame({"actual": y_disc_test * 100, "predicted": disc_pred * 100}),
    x="actual", y="predicted", opacity=0.4,
    title="Predicted vs. actual list-to-sale discount (%)",
    labels={"actual": "Actual list-to-sale discount (%)",
            "predicted": "Predicted list-to-sale discount (%)"},
)
fig.add_shape(type="line", x0=-30, y0=-30, x1=30, y1=30, line=dict(dash="dash"))
fig.update_xaxes(range=[-30, 30])
fig.update_yaxes(range=[-30, 30])
fig.show()


## 🧠 Investment-Committee Takeaways (Part 3)

Suggested bullets for the slide deck:

- Most properties in QCP's footprint sell within N days of listing, but
  buyer-friendly markets like [market] routinely take twice as long.
- After two or more price cuts, a property is materially less likely to ever
  close - these are the listings QCP should approach with low-ball offers.
- A history-aware sale-price model improves MAPE from A% to B% vs. a
  model that uses only the list price and structural features.
- The expected list-to-sale discount model gives QCP a per-listing offer
  recommendation that is grounded in the market's recent behavior.

This concludes the four-credit case study. Your final deliverable should now
include the full pipeline (EDA -> baseline AVM -> NLP-augmented AVM ->
history-aware sale model) and the slide deck synthesizing it for the
Investment Committee.
